# Strong eigenform signatures

Load a bounded strong-signature scan, verify its saved data, and inspect the observed strong-weight and theta-weight bounds. The bounds computed here are bounded-scan evidence; an all-weight conclusion still requires the forward classification theorem.

The ordinary signature is `(weight mod phi(p^m), a_ell1, a_ell2)`. Under `theta^i`, it becomes `(weight + 2i, ell1^i a_ell1, ell2^i a_ell2)` modulo the relevant periods and modulus.

In [15]:
from pathlib import Path
import hashlib
import json
import sys

if hasattr(sys, "set_int_max_str_digits"):
    sys.set_int_max_str_digits(0)

def find_repository_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "strong_signatures").is_dir():
            return candidate
    raise FileNotFoundError("could not find the repository root")

REPOSITORY_ROOT = find_repository_root(Path.cwd())

# Change these parameters to inspect another completed scan.
p = 5
m = 3

SCAN_DIRECTORY = REPOSITORY_ROOT / "strong_signatures" / f"p{p}_m{m}"
EXACT_CACHE_DIRECTORY = REPOSITORY_ROOT / "strong_signatures" / "exact"

# 'summary': status and summary only
# 'scan': additionally verify every per-weight result (recommended)
# 'exact': additionally verify every bound exact-cache file (slowest)
AUDIT_LEVEL = "scan"

# This checks the source certificate SHA-256 for imported weights. It is
# optional because the recorded absolute source path may not exist after
# moving the repository to another machine.
VERIFY_IMPORTED_SOURCE_FILES = False

PYTHON_DIRECTORY = REPOSITORY_ROOT / "python"
if str(PYTHON_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(PYTHON_DIRECTORY))

from strong_signature_bounds import canonical_bounds

print("scan:", SCAN_DIRECTORY)
print("audit level:", AUDIT_LEVEL)

scan: /home/nrustom/Documents/math/hecke-congruences/strong_signatures/p5_m3
audit level: scan


In [16]:
def load_json(path):
    return json.loads(Path(path).read_text())

def compact_json(value):
    return json.dumps(value, ensure_ascii=False, separators=(",", ":"))

def nim_sha1(value):
    return hashlib.sha1(compact_json(value).encode("utf-8")).hexdigest().upper()

def verify_payload_seal(value, path):
    expected = value.get("payload_sha1")
    if expected is None:
        raise AssertionError(f"missing payload seal: {path}")
    payload = {key: item for key, item in value.items() if key != "payload_sha1"}
    actual = nim_sha1(payload)
    if actual != expected.upper():
        raise AssertionError(f"payload seal mismatch: {path}")

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def print_table(headers, rows):
    rows = [[str(value) for value in row] for row in rows]
    widths = [len(str(header)) for header in headers]
    for row in rows:
        widths = [max(width, len(value)) for width, value in zip(widths, row)]
    format_row = lambda row: " | ".join(
        value.ljust(width) for value, width in zip(row, widths)
    )
    print(format_row([str(header) for header in headers]))
    print("-+-".join("-" * width for width in widths))
    for row in rows:
        print(format_row(row))

def build_provisional_summary(scan_directory, status):
    weight_paths = sorted(
        scan_directory.glob("weight_*.json"),
        key=lambda path: int(path.stem.split("_")[1]),
    )
    if not weight_paths:
        raise FileNotFoundError(f"no per-weight results in {scan_directory}")

    representatives = {}
    nonrational = []
    completed_weights = []
    hecke_indices = None
    for path in weight_paths:
        data = load_json(path)
        weight = int(data["weight"])
        completed_weights.append(weight)
        hecke_indices = data["hecke_indices"] if hecke_indices is None else hecke_indices
        assert data["hecke_indices"] == hecke_indices
        for orbit in data["orbits"]:
            for packet in orbit["local_packets"]:
                reference = {
                    "weight": weight,
                    "weight_residue": int(data["weight_residue"]),
                    "orbit": int(orbit["orbit"]),
                    "place": int(packet["place"]),
                    "weight_file": path.name,
                }
                signature = packet["rational_signature"]
                if signature is None:
                    nonrational.append(reference)
                else:
                    key = (data["weight_residue"], tuple(str(value) for value in signature))
                    if key not in representatives:
                        representatives[key] = {
                            **reference,
                            "eigenvalue_residues": [str(value) for value in signature],
                        }

    return {
        "schema": "hecke.strong-signature-summary.v1",
        "prime": int(status["prime"]),
        "exponent": int(status["exponent"]),
        "hecke_indices": hecke_indices,
        "maximum_weight": int(status["maximum_weight"]),
        "completed_weights": completed_weights,
        "bounded_scan_complete": False,
        "rational_signatures": list(representatives.values()),
        "rational_signature_count": len(representatives),
        "nonrational_local_packets": nonrational,
        "nonrational_packet_count": len(nonrational),
        "nonrational_packets_deduplicated": False,
        "all_weight_classification_proved": False,
        "provisional": True,
    }

def load_scan_metadata(scan_directory):
    status = load_json(scan_directory / "status.json")
    summary_path = scan_directory / "summary.json"
    if summary_path.is_file():
        return status, load_json(summary_path), "saved summary.json"
    return status, build_provisional_summary(scan_directory, status), "provisional summary rebuilt from checkpoints"

In [17]:
status, summary, summary_source = load_scan_metadata(SCAN_DIRECTORY)

assert status["schema"] == "hecke.strong-signature-status.v1"
assert summary["schema"] == "hecke.strong-signature-summary.v1"
assert status["prime"] == summary["prime"] == p
assert status["exponent"] == summary["exponent"] == m
assert summary["maximum_weight"] == status["maximum_weight"]
assert status["failed"] == []
if summary["bounded_scan_complete"]:
    assert summary["completed_weights"] == list(range(2, status["maximum_weight"] + 1, 2))
else:
    print("INCOMPLETE SCAN: bounds below use only the saved checkpoints.")

print_table(
    ("field", "value"),
    [
        ("state", status["state"]),
        ("summary source", summary_source),
        ("bounded scan complete", summary["bounded_scan_complete"]),
        ("maximum weight", summary["maximum_weight"]),
        ("completed weights", len(summary["completed_weights"])),
        ("Hecke coordinates", tuple(summary["hecke_indices"])),
        ("rational signatures", summary["rational_signature_count"]),
        ("nonrational packets", summary["nonrational_packet_count"]),
    ],
)

field                 | value             
----------------------+-------------------
state                 | completed         
summary source        | saved summary.json
bounded scan complete | True              
maximum weight        | 698               
completed weights     | 349               
Hecke coordinates     | (2, 19)           
rational signatures   | 1100              
nonrational packets   | 0                 


In [18]:
def audit_scan(scan_directory, exact_cache_directory, level="scan", verify_sources=False):
    if level not in {"summary", "scan", "exact"}:
        raise ValueError("AUDIT_LEVEL must be 'summary', 'scan', or 'exact'")

    status, summary, _ = load_scan_metadata(scan_directory)
    weights = [int(weight) for weight in summary["completed_weights"]]
    report = {
        "weight_files_verified": 0,
        "exact_caches_verified": 0,
        "imported_sources_verified": 0,
        "orbits": 0,
        "local_packets": 0,
        "rational_packets": 0,
        "nonrational_packets": 0,
    }
    rational_representatives = {}
    if level == "summary":
        return report

    expected_schema = "hecke.strong-signatures.v1"
    for weight in weights:
        weight_path = scan_directory / f"weight_{weight}.json"
        data = load_json(weight_path)
        verify_payload_seal(data, weight_path)

        assert data["schema"] == expected_schema
        assert data["weight"] == weight
        assert data["prime"] == status["prime"]
        assert data["exponent"] == status["exponent"]
        assert int(data["modulus"]) == p**m
        assert data["degree"] == weight - 2
        assert data["weight_residue"] == weight % data["weight_period"]
        assert data["degree_residue"] == (weight - 2) % data["weight_period"]
        assert data["hecke_indices"] == summary["hecke_indices"]
        assert data["finite_weight_complete"]

        orbit_dimension = 0
        for orbit in data["orbits"]:
            orbit_dimension += int(orbit["field_degree"])
            local_degree = 0
            for packet in orbit["local_packets"]:
                e = int(packet["ramification_index"])
                f = int(packet["residue_degree"])
                assert packet["krw_ideal_exponent"] == e * (m - 1) + 1
                assert packet["local_embedding_count"] == e * f
                if "p_adic_factor_degree" in packet:
                    assert packet["p_adic_factor_degree"] == e * f
                assert packet["krw_reduction_replayed"]
                local_degree += int(packet["local_embedding_count"])
                signature = packet["rational_signature"]
                if signature is None:
                    report["nonrational_packets"] += 1
                else:
                    assert len(signature) == 2
                    assert all(0 <= int(value) < p**m for value in signature)
                    key = (data["weight_residue"], tuple(str(value) for value in signature))
                    rational_representatives.setdefault(
                        key,
                        (weight, int(orbit["orbit"]), int(packet["place"]), weight_path.name),
                    )
                    report["rational_packets"] += 1
                report["local_packets"] += 1
            assert local_degree == int(orbit["field_degree"])
            report["orbits"] += 1
        assert orbit_dimension == int(data["dimension"])

        provenance = data.get("exact_import_provenance")
        if verify_sources and provenance is not None:
            source = Path(provenance["source_certificate"])
            if not source.is_file():
                raise FileNotFoundError(f"import source is unavailable: {source}")
            assert sha256_file(source) == provenance["source_certificate_sha256"]
            report["imported_sources_verified"] += 1

        if level == "exact":
            exact_path = exact_cache_directory / f"weight_{weight}.json"
            exact = load_json(exact_path)
            verify_payload_seal(exact, exact_path)
            assert nim_sha1(exact) == data["exact_cache_sha1"].upper()
            report["exact_caches_verified"] += 1

        report["weight_files_verified"] += 1

    summary_representatives = {
        (int(row["weight_residue"]), tuple(str(value) for value in row["eigenvalue_residues"])):
        (int(row["weight"]), int(row["orbit"]), int(row["place"]), row["weight_file"])
        for row in summary["rational_signatures"]
    }
    assert rational_representatives == summary_representatives
    assert len(rational_representatives) == summary["rational_signature_count"]
    assert report["nonrational_packets"] == summary["nonrational_packet_count"]
    assert report["rational_packets"] + report["nonrational_packets"] == report["local_packets"]
    report["distinct_rational_signatures"] = len(rational_representatives)
    return report

In [19]:
audit = audit_scan(
    SCAN_DIRECTORY,
    EXACT_CACHE_DIRECTORY,
    level=AUDIT_LEVEL,
    verify_sources=VERIFY_IMPORTED_SOURCE_FILES,
)

print_table(("audit item", "count"), audit.items())
print("DATA AUDIT PASSED")

KeyboardInterrupt: 

## Bounds visible directly in this repository

Rational signatures have canonical pairs in `Z/p^m Z`, so they can be deduplicated and theta-twisted directly. Nonrational packets require a canonical local-algebra presentation before they can be compared across coefficient fields. Consequently, the next cell reports full observed bounds only when every packet is rational; otherwise it reports the rational portion and explicitly marks the full bound as unavailable from `summary.json` alone.

In [20]:
def rational_signature_records(summary):
    records = []
    for row in summary["rational_signatures"]:
        records.append(
            {
                "weight": int(row["weight"]),
                "weight_residue": int(row["weight_residue"]),
                "signature": tuple(int(value) for value in row["eigenvalue_residues"]),
            }
        )
    return records

def rational_theta_orbit_key(record, hecke_indices, modulus, period):
    a1, a2 = record["signature"]
    ell1, ell2 = hecke_indices
    return min(
        (
            (record["weight_residue"] + 2 * i) % period,
            pow(ell1, i, modulus) * a1 % modulus,
            pow(ell2, i, modulus) * a2 % modulus,
        )
        for i in range(period)
    )

records = rational_signature_records(summary)
modulus = p**m
period = (p - 1) * p**(m - 1)
hecke_indices = tuple(int(value) for value in summary["hecke_indices"])

rational_strong_bound = max((row["weight"] for row in records), default=None)
theta_first_weights = {}
for row in records:
    key = rational_theta_orbit_key(row, hecke_indices, modulus, period)
    theta_first_weights[key] = min(theta_first_weights.get(key, row["weight"]), row["weight"])
rational_theta_bound = max(theta_first_weights.values(), default=None)
all_packets_rational = summary["nonrational_packet_count"] == 0

print_table(
    ("quantity", "value", "scope"),
    [
        ("observed strong-weight bound", rational_strong_bound, "all packets" if all_packets_rational else "rational signatures only"),
        ("observed theta-weight bound", rational_theta_bound, "all packets" if all_packets_rational else "rational signatures only"),
        ("ordinary signature count", len(records), "rational signatures"),
        ("theta-orbit count", len(theta_first_weights), "rational signatures"),
    ],
)

if not all_packets_rational:
    print("The canonical calculation below includes the non-rational packets.")

quantity                     | value | scope              
-----------------------------+-------+--------------------
observed strong-weight bound | 598   | all packets        
observed theta-weight bound  | 106   | all packets        
ordinary signature count     | 1100  | rational signatures
theta-orbit count            | 41    | rational signatures


## Weight bounds

In [21]:
bounds = canonical_bounds(SCAN_DIRECTORY, include_theta=True)
last_new_weights = bounds.pop("last_weights_with_new_signatures")
print_table(("canonical quantity", "value"), bounds.items())
print("last new signature weight:", last_new_weights[-1])
print("SELF-CONTAINED CANONICAL BOUNDS COMPUTED")

canonical quantity             | value  
-------------------------------+--------
prime                          | 5      
exponent                       | 3      
modulus                        | 125    
period                         | 100    
hecke_indices                  | (2, 19)
largest_completed_weight       | 698    
rational_packet_occurrences    | 9767   
nonrational_packet_occurrences | 0      
canonical_signature_count      | 1100   
strong_weight_bound            | 598    
canonical_theta_orbit_count    | 41     
theta_weight_bound             | 106    
last new signature weight: 598
SELF-CONTAINED CANONICAL BOUNDS COMPUTED
